In [ ]:
!nvidia-smi
%%capture
!pip install unsloth

In [ ]:
import random, json, hashlib
from collections import Counter

SMELLS = ["Long Parameter List", "Primitive Obsession", "Long Method", "Magic Number Risk"]

SYSTEM = (
    "You predict potential code smells from UML class diagrams written in Mermaid. "
    "Look at the method signatures and attributes. "
    "Possible smells: Long Parameter List, Primitive Obsession, Long Method, Magic Number Risk. "
    "Output only a JSON array of objects with keys 'smell', 'class', 'method' (or 'attribute' if no method), 'reason'. "
    "Output [] if there are no potential code smells."
)

NOUNS = ["Order", "Customer", "Product", "Invoice", "Payment", "User", "Account", "Cart", "Ticket", "Booking"]
ROLES = ["Service", "Repository", "Controller", "Manager"]
VERBS = ["create", "update", "delete", "find", "load", "save", "validate", "send", "calculate", "render"]
ATTRS = ["id", "name", "email", "status", "price", "quantity", "address", "phone", "date", "description", "discount", "tax", "total"]

class D:
    def __init__(s, rng):
        s.r = rng
        s.cls = {}
        s.counter = 0
    def name(s, prefix="Class"):
        s.counter += 1
        return f"{s.r.choice(NOUNS)}{s.r.choice(ROLES)}_{s.counter}"
    def add(s, attrs=None, ops=None):
        r = s.r
        n = s.name()
        
        # Attributes
        a = []
        if attrs is not None:
            for item in attrs:
                a.append(item)
        else:
            for _ in range(r.randint(0, 3)):
                vis = r.choice("+-")
                a.append(f"{vis}{r.choice(['String', 'int', 'double', 'boolean'])} {r.choice(ATTRS)}")
                
        # Operations
        o = []
        if ops is not None:
            for item in ops:
                o.append(item)
        else:
            for _ in range(r.randint(0, 4)):
                o.append(f"+{r.choice(VERBS)}{r.choice(NOUNS)}()")
                
        s.cls[n] = {"a": a, "o": o}
        return n

    def render(s):
        r = s.r
        lines = ["classDiagram"]
        names = list(s.cls)
        for n in names:
            c = s.cls[n]
            if c["a"] or c["o"]:
                lines.append(f"  class {n} {{")
                for m in c["a"] + c["o"]: lines.append(f"    {m}")
                lines.append("  }")
            else:
                lines.append(f"  class {n}")
        return "\n".join(lines)

# Injectors
def inject_long_param(d):
    c = d.add(ops=["+processOrder(customerName: String, customerEmail: String, productId: int, quantity: int, price: double, discountCode: String, shippingAddress: String)"])
    
def inject_primitive(d):
    c = d.add(attrs=["+String email", "+String phone", "+String zipCode", "+String currency"])

def inject_long_method(d):
    c = d.add(ops=["+validateAndProcessAndSaveAndNotifyOrder()", "+generateReportAndSendEmailAndLogAction()"])

def inject_magic_number(d):
    c = d.add(attrs=["+double taxRate", "+double discountPercentage", "+int maxRetryCount"])

def make_example(rng):
    d = D(rng)
    # 1. Add some clean background classes
    for _ in range(rng.randint(1, 4)):
        d.add()
    
    # 2. Pick which smells to inject
    k = rng.choices([0, 1, 2], [0.3, 0.5, 0.2])[0]
    chosen = rng.sample(SMELLS, k)
    
    # 3. Inject
    if "Long Parameter List" in chosen: inject_long_param(d)
    if "Primitive Obsession" in chosen: inject_primitive(d)
    if "Long Method" in chosen: inject_long_method(d)
    if "Magic Number Risk" in chosen: inject_magic_number(d)
    
    # 4. Generate Labels (Mock Ground Truth based on the rules)
    labels = []
    for name, c in d.cls.items():
        for op in c["o"]:
            # Long Parameter List
            if op.count(",") >= 5:
                labels.append({"smell": "Long Parameter List", "class": name, "method": op.split("(")[0][1:], "reason": "Method has more than 5 parameters."})
            # Long Method
            if "And" in op or op.count("And") >= 2:
                labels.append({"smell": "Long Method", "class": name, "method": op.split("(")[0][1:], "reason": "Method name implies multiple responsibilities."})
        
        for attr in c["a"]:
            # Primitive Obsession
            if "String email" in attr or "String phone" in attr or "String zipCode" in attr or "String currency" in attr:
                labels.append({"smell": "Primitive Obsession", "class": name, "attribute": attr.split()[-1], "reason": "Domain concept represented as a raw String."})
            # Magic Number Risk
            if "double tax" in attr or "double discount" in attr or "int maxRetry" in attr:
                labels.append({"smell": "Magic Number Risk", "class": name, "attribute": attr.split()[-1], "reason": "Business logic values should be dedicated types, not raw numbers."})
                
    # Deduplicate
    unique_labels = []
    for l in labels:
        if l not in unique_labels: unique_labels.append(l)
        
    if len(d.cls) > 20: return None
    return d.render(), unique_labels

def build(n_train=2000, n_val=100, n_test=200, seed=42):
    rng = random.Random(seed)
    seen = set()
    data_ = []
    need = n_train + n_val + n_test
    while len(data_) < need:
        ex = make_example(rng)
        if not ex: continue
        h = hashlib.md5(ex[0].encode()).hexdigest()
        if h in seen: continue
        seen.add(h)
        data_.append(ex)
        
    def fmt(ex):
        return {"messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": ex[0]},
            {"role": "assistant", "content": json.dumps(ex[1])}
        ]}
    
    rows = [fmt(e) for e in data_]
    return rows[:n_train], rows[n_train:n_train+n_val], rows[n_train+n_val:]

In [ ]:
import collections
train_rows, val_rows, test_rows = build(n_train=2000, n_val=100, n_test=200)

for name, rows in [("code_smell_train", train_rows), ("code_smell_val", val_rows), ("code_smell_test", test_rows)]:
    with open(f"{name}.jsonl", "w") as f:
        for r in rows: f.write(json.dumps(r) + "\n")

cnt = collections.Counter()
clean = 0
for r in train_rows:
    lab = json.loads(r["messages"][2]["content"])
    clean += (not lab)
    for s in {x["smell"] for x in lab}: cnt[s] += 1

print(f"train={len(train_rows)} val={len(val_rows)} test={len(test_rows)} | clean diagrams: {clean}")
print("Smells present in train:", dict(cnt))
print("\n--- Example Diagram ---")
print(train_rows[5]["messages"][1]["content"])
print("\n--- Ground Truth ---")
print(train_rows[5]["messages"][2]["content"])

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Shorter than design smells, we just look at signatures
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth", random_state = 3407,
)

In [ ]:
from datasets import Dataset

def to_text(rows):
    return Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize=False)} for r in rows])

train_ds = to_text(train_rows)
print("Data formatted.")

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = train_ds,
    dataset_text_field = "text", max_seq_length = max_seq_length, packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
        num_train_epochs = 2, learning_rate = 2e-4, warmup_steps = 10,
        lr_scheduler_type = "linear", optim = "adamw_8bit", weight_decay = 0.01,
        fp16 = not is_bfloat16_supported(), bf16 = is_bfloat16_supported(),
        logging_steps = 10, save_strategy = "no", report_to = "none",
        seed = 3407, output_dir = "outputs",
    ),
)
trainer = train_on_responses_only(
    trainer, instruction_part = "<|im_start|>user\n", response_part = "<|im_start|>assistant\n")
stats = trainer.train()
print(f"Training time: {stats.metrics['train_runtime']/60:.1f} min")

In [ ]:
import json, torch
from collections import defaultdict
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"

def predict(users, bs=8, max_new=300):
    order = sorted(range(len(users)), key=lambda i: len(users[i])); res = [None]*len(users)
    for s in range(0, len(order), bs):
        idx = order[s:s+bs]
        prompts = [tokenizer.apply_chat_template(
            [{"role":"system","content":SYSTEM},{"role":"user","content":users[i]}],
            tokenize=False, add_generation_prompt=True) for i in idx]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True)
        for i, o in zip(idx, tokenizer.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)):
            res[i] = o.strip()
    return res

def parse(s):
    try:
        v = json.loads(s)
        parsed = []
        for x in v:
            loc = x.get("method", x.get("attribute", ""))
            parsed.append((x["smell"], x["class"], loc))
        return parsed, True
    except Exception:
        return [], False

users = [r["messages"][1]["content"] for r in test_rows]
gold  = [parse(r["messages"][2]["content"])[0] for r in test_rows]
preds = predict(users)

tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
exact = invalid = 0; clean_ok = clean_tot = 0

for p_raw, g in zip(preds, gold):
    p, ok = parse(p_raw); invalid += (not ok)
    gk = set(g); pk = set(p)
    exact += (gk == pk)
    if not g: clean_tot += 1; clean_ok += (not p)
    for k in pk:
        if k in gk: tp[k[0]] += 1
        else: fp[k[0]] += 1
    for k in gk:
        if k not in pk: fn[k[0]] += 1

print(f"Invalid JSON: {invalid}/{len(preds)}")
print(f"Exact match (whole diagram): {exact/len(preds):.1%}")
print(f"Clean diagrams correctly answered []: {clean_ok}/{clean_tot}")
print(f"\n{'Smell':26s} {'P':>6s} {'R':>6s} {'F1':>6s}")
for s in SMELLS:
    P = tp[s]/max(tp[s]+fp[s],1); R = tp[s]/max(tp[s]+fn[s],1); F = 2*P*R/max(P+R,1e-9)
    print(f"{s:26s} {P:6.2f} {R:6.2f} {F:6.2f}")

In [ ]:
my_diagram = """classDiagram
  class CheckoutService {
    +processPayment(cardNumber: String, expiry: String, cvv: String, amount: double, currency: String, billingAddress: String, shippingAddress: String)
    +double taxRate
  }
"""
print("Prediction:", predict([my_diagram])[0])

model.save_pretrained("code_smell_lora")
tokenizer.save_pretrained("code_smell_lora")
!zip -r code_smell_lora.zip code_smell_lora
from google.colab import files
files.download("code_smell_lora.zip")